# WCO Forecasting — LSTM Training (Google Colab)

Train the LSTM here, **separately from the web app**. Free GPU, no install pain.

**Workflow**
1. Export your seeded/real WCO data to a CSV with columns: `establishment_id, week_date, quantity_liters`.
2. Upload it to this notebook (or mount Google Drive).
3. Run all cells.
4. Download `lstm_wco.pt` and `scaler.npz`.
5. Drop both files into `backend/app/services/model_artifacts/`.

The model class below **must stay identical** to the one in
`backend/app/services/forecasting.py` (same `SEQUENCE_LENGTH`, `HIDDEN_SIZE`, `NUM_LAYERS`).

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# These three constants MUST match backend/app/services/forecasting.py
SEQUENCE_LENGTH = 12
HIDDEN_SIZE = 64
NUM_LAYERS = 2

## 2. Load data

To export from your backend, run this once in the `backend/` folder:

```python
from app.core.database import SessionLocal
from app.models.wco import WCOGenerationRecord
import pandas as pd
from sqlalchemy import select
db = SessionLocal()
rows = db.scalars(select(WCOGenerationRecord)).all()
pd.DataFrame([{'establishment_id': r.establishment_id, 'week_date': r.week_date, 'quantity_liters': r.quantity_liters} for r in rows]).to_csv('wco_export.csv', index=False)
```

Then upload `wco_export.csv` here.

In [ ]:
from google.colab import files
uploaded = files.upload()  # choose wco_export.csv
df = pd.read_csv('wco_export.csv', parse_dates=['week_date'])
df = df.sort_values(['establishment_id', 'week_date'])
print(df.shape)
df.head()

## 3. Build training sequences

We scale all volumes to [0, 1] with a single global min/max (saved as `scaler.npz`
so the backend inverse-scales the same way), then slide a window of
`SEQUENCE_LENGTH` weeks across each establishment's series to make (X, y) pairs.

In [ ]:
vals = df['quantity_liters'].to_numpy(dtype=np.float32)
data_min, data_max = float(vals.min()), float(vals.max())
span = (data_max - data_min) or 1.0
print('min', data_min, 'max', data_max)

X_list, y_list = [], []
for est_id, g in df.groupby('establishment_id'):
    series = ((g['quantity_liters'].to_numpy(dtype=np.float32) - data_min) / span)
    for i in range(len(series) - SEQUENCE_LENGTH):
        X_list.append(series[i:i + SEQUENCE_LENGTH])
        y_list.append(series[i + SEQUENCE_LENGTH])

X = np.array(X_list, dtype=np.float32)[..., None]  # (N, seq, 1)
y = np.array(y_list, dtype=np.float32)[..., None]  # (N, 1)
print('sequences:', X.shape, y.shape)

# Chronological-ish split: 80% train, 20% validation
n = len(X)
idx = int(n * 0.8)
X_train, y_train = X[:idx], y[:idx]
X_val, y_val = X[idx:], y[idx:]

train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
val_ds = TensorDataset(torch.tensor(X_val), torch.tensor(y_val))
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=128)

## 4. Model (keep identical to the backend)

In [ ]:
class WCOLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

model = WCOLSTM().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
print(model)

## 5. Train (with early stopping)

In [ ]:
EPOCHS = 80
PATIENCE = 8
best_val = float('inf')
patience_left = PATIENCE
best_state = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        vloss = np.mean([
            criterion(model(xb.to(device)), yb.to(device)).item()
            for xb, yb in val_dl
        ])
    print(f'epoch {epoch:3d}  val_mse {vloss:.5f}')

    if vloss < best_val - 1e-5:
        best_val = vloss
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_left = PATIENCE
    else:
        patience_left -= 1
        if patience_left == 0:
            print('Early stopping.')
            break

model.load_state_dict(best_state)

## 6. Evaluate (MAE, RMSE, R² in real liters)

In [ ]:
model.eval()
with torch.no_grad():
    pred = model(torch.tensor(X_val).to(device)).cpu().numpy().ravel()
true = y_val.ravel()

# inverse-scale back to liters
pred_l = pred * span + data_min
true_l = true * span + data_min

mae = mean_absolute_error(true_l, pred_l)
rmse = mean_squared_error(true_l, pred_l) ** 0.5
r2 = r2_score(true_l, pred_l)
print(f'MAE  {mae:.3f} L')
print(f'RMSE {rmse:.3f} L')
print(f'R2   {r2:.4f}')
print('\nReport these metrics in your thesis. If R2 is poor, see the ARIMA fallback note in the README.')

## 7. Export weights + scaler for the backend

In [ ]:
torch.save(model.state_dict(), 'lstm_wco.pt')
np.savez('scaler.npz', min=data_min, max=data_max)
print('Saved lstm_wco.pt and scaler.npz')

from google.colab import files
files.download('lstm_wco.pt')
files.download('scaler.npz')
print('Place both files in backend/app/services/model_artifacts/')